# Tag Recommendation Experiment (RecSys 2018)

Notebook simple et modulaire pour reproduire une expérience inspirée de l'article **Semantic-based Tag Recommendation in Scientific Bookmarking Systems**.

- Import des données : **1 cellule**
- Une fonction par cellule
- Visualisations déplacées dans `src/visualization.py`
- Dernière cellule : compilation des résultats + visualisations comparatives

In [ ]:
# Cellule 1 - Imports (meme stack que l'article)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data import load_simple_dataset, keep_top_k_tags, preprocess_text_nltk, text_length_stats
from src.experiment import prepare_train_test, run_all_models
from src.visualization import (
    plot_tag_distribution,
    plot_text_length_distributions,
    plot_model_metrics,
    plot_article_vs_current,
)

pd.set_option("display.max_colwidth", 120)

In [ ]:
# Cellule 2 - Paramètres globaux
DATA_PATH = str(PROJECT_ROOT / "data" / "citeulike_top10.csv")
GLOVE_PATH = str(PROJECT_ROOT / "data" / "glove.6B.300d.txt")  # optionnel
TOP_K_TAGS = 10
TEST_SIZE = 0.1
RANDOM_STATE = 42

In [ ]:
# Cellule 3 - Import des données (simple, en une cellule)
def import_data(csv_path: str, top_k_tags: int = 10):
    df = load_simple_dataset(csv_path)
    df = keep_top_k_tags(df, top_k=top_k_tags)
    df = preprocess_text_nltk(df)
    return text_length_stats(df)


df = import_data(DATA_PATH, TOP_K_TAGS)
df.head()

In [ ]:
# Cellule 4 - Fonction de résumé exploration
def summarize_dataset(df: pd.DataFrame) -> pd.DataFrame:
    summary = {
        "num_documents": [len(df)],
        "num_unique_tags": [len({t for tags in df["tag_list"] for t in tags})],
        "avg_title_words": [df["title_words"].mean()],
        "avg_abstract_words": [df["abstract_words"].mean()],
        "avg_tags_per_doc": [df["num_tags"].mean()],
    }
    return pd.DataFrame(summary)


summarize_dataset(df)

In [ ]:
# Cellule 5 - Fonction visualisation EDA
def visualize_eda(df: pd.DataFrame):
    fig1 = plot_tag_distribution(df, top_n=TOP_K_TAGS)
    fig2 = plot_text_length_distributions(df)
    plt.show()
    return fig1, fig2


_ = visualize_eda(df)

In [ ]:
# Cellule 6 - Préparation train/test
def make_splits(df: pd.DataFrame):
    return prepare_train_test(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )


X_train, X_test, y_train, y_test, mlb = make_splits(df)
print("Train:", len(X_train), "Test:", len(X_test), "Labels:", len(mlb.classes_))

In [ ]:
# Cellule 7 - Entraînement de tous les modèles
def train_models(X_train, X_test, y_train, y_test, n_topics: int, glove_path: str):
    return run_all_models(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        n_topics=n_topics,
        glove_path=glove_path,
    )


metrics_df, predictions = train_models(
    X_train,
    X_test,
    y_train,
    y_test,
    n_topics=TOP_K_TAGS,
    glove_path=GLOVE_PATH,
)
metrics_df

In [ ]:
# Cellule 8 - Sauvegarde facultative des métriques
def save_metrics(metrics_df: pd.DataFrame, output_csv: str = "data/metrics_results.csv"):
    metrics_df.to_csv(output_csv, index=False)
    return output_csv


save_metrics(metrics_df)

In [ ]:
# Cellule 9 (dernière) - Compilation + visualisations comparatives
def compile_and_compare(metrics_df: pd.DataFrame):
    display(metrics_df.sort_values("micro_f1", ascending=False).reset_index(drop=True))

    fig_models = plot_model_metrics(metrics_df)
    fig_article = plot_article_vs_current(metrics_df)
    plt.show()

    return {
        "best_model": metrics_df.sort_values("micro_f1", ascending=False).iloc[0]["name"],
        "best_micro_f1": float(metrics_df["micro_f1"].max()),
    }


summary = compile_and_compare(metrics_df)
summary